# Bookstore Sales Analysis
Dataset: 200 bookstore transactions from 2025. Goal: clean the data, summarize sales by category and month, describe transaction values, and find potential outliers with the IQR method.

In [ ]:
import pandas as pd

df = pd.read_csv("bookstore_sales.csv")
df.head()

## 1. Inspect the data

In [ ]:
df.info()
print("Duplicate rows:", df.duplicated().sum())
print(df["category"].unique())

## 2. Clean and prepare the data

In [ ]:
df = df.drop_duplicates()
df = df.dropna(subset=["quantity"])
df["category"] = df["category"].str.strip().str.title()
df["date"] = pd.to_datetime(df["date"])
df["total"] = df["quantity"] * df["unit_price"]
df["month"] = df["date"].dt.to_period("M")
print(df.shape)
print(df["category"].unique())

## 3. Sales by category

In [ ]:
by_category = df.groupby("category").agg(
    transactions=("transaction_id", "count"),
    units=("quantity", "sum"),
    revenue=("total", "sum"),
).sort_values("revenue", ascending=False).round(2)
by_category

In [ ]:
import matplotlib.pyplot as plt

by_category["revenue"].plot(kind="bar", color="steelblue", edgecolor="black")
plt.title("Revenue by category")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=0)
plt.show()

## 4. Sales by month

In [ ]:
by_month = df.groupby("month").agg(
    transactions=("transaction_id", "count"),
    revenue=("total", "sum"),
).round(2)
by_month

## 5. Descriptive statistics

In [ ]:
df["total"].describe().round(2)

In [ ]:
df["total"].plot(kind="hist", bins=20, color="steelblue", edgecolor="black")
plt.title("Distribution of transaction amounts")
plt.xlabel("Transaction amount (USD)")
plt.ylabel("Number of transactions")
plt.show()

## 6. Outliers with the IQR method

In [ ]:
q1, q3 = df["total"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print("IQR:", round(iqr, 2), "| Lower bound:", round(lower, 2), "| Upper bound:", round(upper, 2))
outliers = df[(df["total"] < lower) | (df["total"] > upper)]
print("Potential outliers:", len(outliers))
outliers.sort_values("total", ascending=False)

## 7. Findings
- **Cleaning:** 201 rows became 197 transactions. We removed 1 duplicate row and 3 rows with no quantity, and fixed 11 category spellings down to 6 categories.
- **Categories:** Fiction leads revenue (about $1,823 from 62 transactions), followed by Children (about $1,468) and Non-Fiction (about $1,356). Comics brings in the least (about $490).
- **Months:** December is the best month (about $1,072). June and September are high because of large bulk orders. February and May are the weakest months (under $200).
- **Statistics:** the average transaction is about $33.66, but the median is only $22.71, so a few large orders pull the average up.
- **Outliers:** the IQR rule flags 11 transactions above $73.96. The three biggest are bulk orders: 25 children's books ($507.50), 12 science books ($218.16), and 15 fiction books ($178.65). They are real sales, not errors, so we keep them.